In [47]:
import pandas as pd
import numpy as np

Vds_max = 12
Vth = 3
Ta = 25 # Temperatura ambiente en grados Celsius
Rt_jc = 1.1 # grados C / W
Rt_cs = 0.5 # grados C / W
Rt_ja = 62 # grados C / W

Rt_ca = Rt_ja - Rt_jc - Rt_cs
print("Resistencia termica del disipador:", Rt_ca, "C/W")

r_norm = pd.Series([1000, 1500, 2200, 3300, 4700, 5100, 6800])
r1 = r_norm

Resistencia termica del disipador: 60.4 C/W


## Calculo de disipador del IRF540N
[datasheet](https://www.farnell.com/datasheets/67691.pdf)

In [51]:
df_disipador = pd.DataFrame([250e-3, 350e-3, 500e-3, 750e-3], columns=['Imax [A]'])
df_disipador["Pdis [W]"] = df_disipador["Imax [A]"] * Vds_max

df_disipador["Tj"] = Ta + df_disipador["Pdis [W]"] * Rt_ja

Rth_sa_2225 = 15 + Rt_cs# grados C / W
Rth_eq_2225 = Rt_ca * Rth_sa_2225 / (Rt_ca + Rth_sa_2225)
print(f"Resistencia termica equivalente del disipador: {Rth_eq_2225:.2f}C/W")

Rth_sa_2725 = 3.5 + Rt_cs# grados C / W
Rth_eq_2725 = Rt_ca * Rth_sa_2725 / (Rt_ca + Rth_sa_2725)
print(f"Resistencia termica equivalente del disipador: {Rth_eq_2725:.2f}C/W")


df_disipador["Tj_eq_2225"] = Ta + df_disipador["Pdis [W]"] * Rth_eq_2225
df_disipador["Tj_eq_2725"] = Ta + df_disipador["Pdis [W]"] * Rth_eq_2725

df_disipador


Resistencia termica equivalente del disipador: 12.33C/W
Resistencia termica equivalente del disipador: 3.75C/W


,Imax [A],Pdis [W],Tj,Tj_eq_2225,Tj_eq_2725
0,0.25,3.0,211.0,62.003953,36.254658
1,0.35,4.2,285.4,76.805534,40.756522
2,0.50,6.0,397.0,99.007905,47.509317
3,0.75,9.0,583.0,136.011858,58.763975


## Op Amp Calculos

In [52]:
Vgs_max = 16
v_lpf_max = 3.3
gain_amp = Vgs_max / v_lpf_max
print(f"Ganacia objetivo: {gain_amp:.2f}")
df = pd.DataFrame(r_norm, columns=['R1_norm'])
df["R2"] = (gain_amp - 1) * df["R1_norm"]
display(df)

vout_test = v_lpf_max * (1 + 20e3 / 5.1e3)
vout_test

Ganacia objetivo: 4.85


,R1_norm,R2
0,1000,3848.484848
1,1500,5772.727273
2,2200,8466.666667
3,3300,12700.000000
4,4700,18087.878788
5,5100,19627.272727
6,6800,26169.696970


16.241176470588236

## Fuente Boost para Op Amp
Se utiliza el [MC34063](https://www.ti.com/lit/ds/symlink/mc34063a.pdf)


In [19]:
v_in = 5.0      # Voltaje de entrada
v_out = 19.0    # Voltaje de salida
i_out = 0.5     # Corriente de salida
f0 = 25 * 1000  # Frecuencia de operación
t0 = 1 / f0     # Periodo de operación
v_ripple = 0.1  # Voltaje de rizado

$\frac{ton}{toff}=A$ -> $ton=A*(T - ton)$ -> $ton = \frac{A*T}{1+A}$


In [20]:
def mc34063_calculations(v_in, v_out, i_out, f0, v_ripple):
    """
    Calcula los componentes necesarios para el circuito boost con MC34063.
    """
    VF = 0.4
    VSAT = 1.42
    t0 = 1 / f0
    ton_toff = (v_out + VF - v_in) / (v_in - VSAT)
    ton = (ton_toff * t0) / (1 + ton_toff)
    toff = t0 - ton
    cap_t = 40e-6 * ton
    ipk = 2 * i_out * (ton_toff + 1)
    r_sc = 0.3 / ipk

    l_min = ton * ((v_in - VSAT) / ipk)

    cap_o = 9 * i_out * ton / v_ripple    
    return {
        "duty": ton / t0,
        "ton": ton,
        "toff": toff,
        "cap_t": cap_t,
        "ipk": ipk,
        "r_sc": r_sc,
        "l_min": l_min,
        "cap_o": cap_o
    }

mc34063_calculations(v_in, v_out, i_out, f0, v_ripple)

{'duty': 0.8008898776418243,
 'ton': 3.203559510567297e-05,
 'toff': 7.96440489432703e-06,
 'cap_t': 1.281423804226919e-09,
 'ipk': 5.022346368715083,
 'r_sc': 0.05973303670745273,
 'l_min': 2.2835428315480933e-05,
 'cap_o': 0.0014416017797552838}

In [ ]:
r2_calc = lambda r1: (v_out / 1.25 - 1) * r1

r2 = r1.apply(r2_calc)
pd.DataFrame({
    "R1 (Ohm)": r1,
    "R2 (Ohm)": r2
}).set_index("R1 (Ohm)").round(2)

,R2 (Ohm)
R1 (Ohm),
1000,14200.0
1500,21300.0
2200,31240.0
3300,46860.0
4700,66740.0
5100,72420.0
6800,96560.0


In [ ]:
r1 = 3300
r2 = 47000
v_out_calc = 1.25 * (1 + r2 / r1)
v_out_calc

19.335106382978722